In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [2]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking,number_of_functions
id,,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,hotel,NaN,Hotel,7,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
2,890729731,Świętego Piotra,6D,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
3,890292040,Węglowa,22,NaN,centrum handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
4,890729732,Świętego Piotra,6,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
5,136885205,Tadeusza Wendy,15,NaN,centrum medyczne,NaN,Centrum Medyczne,6,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,2,2026-09-27,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1237,126180359,Władysława IV,61,NaN,"laboratorium medyczne,piekarnia,kiosk",NaN,Mieszkaniowa Wielorodzinna,7,<NA>,1940,modernizm,3,NaN,NaN,google.com/maps,2,2026-09-30,3
1238,126173503,Świętojańska,122,NaN,"apteka,handel zdrowie",NaN,Biurowa,6,<NA>,1930,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-30,2
1239,260676369,Aleja Marszałka Józefa Piłsudskiego,56,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,7,<NA>,1930,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-30,1


In [3]:
data_gdynia_long = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long.head()

,addr_street,ground_floor_function
id,,
1,Świętego Piotra,hotel
2,Świętego Piotra,mieszkaniowa wielorodzinna
3,Węglowa,centrum handlowe
4,Świętego Piotra,mieszkaniowa wielorodzinna
5,Tadeusza Wendy,centrum medyczne


1.

In [4]:
def storeys_range(s):
    return s.max() - s.min()


data_gdynia_starowiejska_num_storeys = data_gdynia.loc[
    data_gdynia["addr_street"] == "Starowiejska", "number_of_storeys"
]
storeys_range(data_gdynia_starowiejska_num_storeys)

np.int64(6)

In [5]:
data_dynia_max_min_func = data_gdynia.groupby('addr_street')["number_of_storeys"].agg(storeys_range)

data_gdynia_max_min_diff = data_gdynia.groupby("addr_street").agg(
    max_storeys=("number_of_storeys", "max"), min_storeys=("number_of_storeys", "min")
).assign(storeys_range=lambda d: d["max_storeys"] - d["min_storeys"])

data_gdynia_max_min_lambda = data_gdynia.groupby("addr_street")["number_of_storeys"].agg(
    lambda s: s.max() - s.min())

In [6]:
assert (data_dynia_max_min_func).equals(data_gdynia_max_min_diff.loc[:, 'storeys_range'])
assert (data_dynia_max_min_func).equals(data_gdynia_max_min_lambda)

2.

In [7]:
def diff_from_median(s):
    return s - s.median()

print(len(diff_from_median(data_gdynia_starowiejska_num_storeys)))
diff_from_median(data_gdynia_starowiejska_num_storeys).head()


59


id
276   -4.0
302   -3.0
309   -2.0
314   -4.0
318   -4.0
Name: number_of_storeys, dtype: Float64

In [8]:
data_gdynia_diff_median = data_gdynia.assign(storeys_vs_median = data_gdynia.groupby('addr_street')['number_of_storeys'].transform(diff_from_median))
data_gdynia_diff_median = data_gdynia_diff_median.loc[:,['addr_street','number_of_storeys','storeys_vs_median' ]]

In [9]:
assert np.isclose(data_gdynia_diff_median.groupby('addr_street')['storeys_vs_median'].agg('median'), 0).all()

3.

In [10]:
def share_tall(s, threshold):
    return (s >= threshold).mean()

share_tall(data_gdynia_starowiejska_num_storeys, 5)

np.float64(0.6779661016949152)

In [11]:
data_gdynia_shares_threshold = data_gdynia.groupby('addr_street').agg(
    share_5 = ('number_of_storeys', lambda d: share_tall(d,5)),
    share_8 = ('number_of_storeys', lambda d: share_tall(d,8)),
)
data_gdynia_shares_threshold.head(10)

,share_5,share_8
addr_street,,
10 Lutego,0.5,0.020833
3 Maja,0.5,0.153846
Aleja Jana Pawła II,0.066667,0.0
Aleja Marszałka Józefa Piłsudskiego,1.0,0.0
Antoniego Abrahama,0.7,0.0
Antoniego Hryniewickiego,0.12,0.0
Arkadiusza Rybickiego,0.666667,0.0
Armii Krajowej,0.694444,0.027778
Dworcowa,0.2,0.0


In [12]:
assert len(data_gdynia_shares_threshold) == data_gdynia['addr_street'].nunique()
assert (data_gdynia_shares_threshold['share_5'] >= data_gdynia_shares_threshold['share_8']).all()

In [13]:
data_gdynia_threshold_5 = data_gdynia.groupby('addr_street')['number_of_storeys'].agg(share_tall, 5)
data_gdynia_threshold_8 = data_gdynia.groupby('addr_street')['number_of_storeys'].agg(share_tall, 8)

In [14]:
pd.DataFrame({"share_5": data_gdynia_threshold_5, "share_8": data_gdynia_threshold_8}).head(10)

,share_5,share_8
addr_street,,
10 Lutego,0.5,0.020833
3 Maja,0.5,0.153846
Aleja Jana Pawła II,0.066667,0.0
Aleja Marszałka Józefa Piłsudskiego,1.0,0.0
Antoniego Abrahama,0.7,0.0
Antoniego Hryniewickiego,0.12,0.0
Arkadiusza Rybickiego,0.666667,0.0
Armii Krajowej,0.694444,0.027778
Dworcowa,0.2,0.0


4.

In [15]:
table_starowiejska = data_gdynia.groupby('addr_street').get_group('Starowiejska')

In [16]:
def street_profile(group):
      tallest_id = group['number_of_storeys'].idxmax()
      return pd.Series([len(group),group['number_of_storeys'].loc[tallest_id], group['arch_style'].loc[tallest_id]],index = ['num_build', 'max_storeys', 'tallest_style'])

street_profile(table_starowiejska)


num_build               59
max_storeys              7
tallest_style    modernizm
dtype: object

In [17]:
data_gdynia_street_profile = data_gdynia.groupby('addr_street').apply(street_profile)

In [18]:
assert len(data_gdynia_street_profile) == data_gdynia['addr_street'].nunique()

assert (data_gdynia_street_profile['num_build']).equals(data_gdynia.groupby('addr_street')['osm_id'].size())

assert data_gdynia_street_profile['num_build'].sum() == len(data_gdynia) - data_gdynia['addr_street'].isna().sum()

assert ((data_gdynia_street_profile['max_storeys']) ==(data_gdynia.groupby('addr_street')['number_of_storeys'].agg('max'))).all()

5.

In [19]:
def only_big_streets(df, min_build):
    return df.loc[df.groupby('addr_street').transform('size') >= min_build]

data_gdynia_big_streets_20 = data_gdynia.pipe(only_big_streets, min_build = 20)
data_gdynia_big_streets_10 = data_gdynia.pipe(only_big_streets, min_build = 10)

In [20]:
assert data_gdynia_big_streets_20.equals(only_big_streets(data_gdynia, 20))
assert len(data_gdynia_big_streets_20) == 605
assert data_gdynia_big_streets_20['addr_street'].nunique() == 13
assert len(data_gdynia_big_streets_10) == 754
assert data_gdynia_big_streets_10['addr_street'].nunique() == 25
assert data_gdynia_big_streets_10['addr_street'].isna().sum() == 0

In [21]:
data_gdynia.pipe(only_big_streets, min_build = 20).groupby('addr_street')['number_of_storeys'].agg(storeys_range).sort_values(ascending = False)

addr_street
Armii Krajowej               12
3 Maja                       10
Władysława IV                10
Skwer Tadeusza Kościuszki    10
Jerzego Waszyngtona          10
Wójta Radtkego                9
Świętojańska                  8
10 Lutego                     7
Jana z Kolna                  7
Antoniego Hryniewickiego      6
Antoniego Abrahama            6
Starowiejska                  6
Stefana Żeromskiego           6
Name: number_of_storeys, dtype: Int64